**Alkebulan Bantu-Luganda Tokenizer (ABLT)**



A morphologically-aware tokenizer designed for Luganda and other Bantu languages. This notebook guides you through:

Environment Setup: Installing required libraries.

Data Access: Instructions for uploading your Luganda corpus.

Resource Loading: Importing phonological and lexical resources.

Morphological Analysis: Extracting noun class markers, verbal extensions, and other patterns.

Tokenizer Training: Configuring and training a customized SentencePiece model.

Evaluation & Comparison: Benchmarking the enhanced tokenizer against a standard model.

Integration: Packaging the tokenizer for use with Transformers and Simpson-based attention mechanisms.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

# Check for resource files in RunPod environment
def check_files():
    required_files = {
        "MergedTok.txt": "Luganda corpus",
        "allophones.lg.xml": "Phonological features",
        "lg_lexicon.dict": "Luganda lexicon"
    }

    found_files = {}
    missing_files = []

    for filename, description in required_files.items():
        if os.path.exists(filename):
            size_mb = os.path.getsize(filename) / (1024 * 1024)
            found_files[filename] = f"{description} ({size_mb:.2f} MB)"
        else:
            missing_files.append(f"{filename} ({description})")

    return found_files, missing_files

# Run the check
found_files, missing_files = check_files()

# Print results
if missing_files:
    print("WARNING: Some required files are missing:")
    for file in missing_files:
        print(f"  - {file}")
    print("\nPlease ensure all files are present in the current directory.")
else:
    print("All required files found!")

for filename, description in found_files.items():
    print(f"  - {filename}: {description}")

# Set the corpus path
corpus_path = "MergedTok.txt"

# Preview corpus content if available
if "MergedTok.txt" in found_files:
    print("\nPreview of corpus content:")
    with open(corpus_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for i, line in enumerate(f) if i < 3]
    for i, line in enumerate(lines, 1):
        preview = line[:80] + ("..." if len(line) > 80 else "")
        print(f"  Line {i}: {preview}")

All required files found!
  - MergedTok.txt: Luganda corpus (5.91 MB)
  - allophones.lg.xml: Phonological features (0.01 MB)
  - lg_lexicon.dict: Luganda lexicon (0.08 MB)

Preview of corpus content:
  Line 1: Abanoonyiboobubudamu bonna baasabiddwa beewandiise ewa ssentebe.
  Line 2: Baayise olukungaana lw'abanoonyiboobubudamu eggulo.
  Line 3: Abanoonyiboobubudamu b'abadde n'obutakkaanya wakati waabwe.


In [ ]:
import os
import re
import json
from xml.etree import ElementTree as ET

# Functions to load resources.
def load_phonological_features(xml_path="allophones.lg.xml"):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        vowels = {}
        for vowel in root.findall(".//vowel"):
            vowels[vowel.get('ph')] = {
                'length': 'long' if vowel.get('vlng') == 'l' else 'short',
                'height': vowel.get('vheight'),
                'front': vowel.get('vfront'),
                'round': True if vowel.get('vrnd') == '+' else False
            }
        consonants = {}
        for consonant in root.findall(".//consonant"):
            consonants[consonant.get('ph')] = {
                'type': consonant.get('ctype'),
                'place': consonant.get('cplace'),
                'voiced': True if consonant.get('cvox') == '+' else False,
                'long': True if consonant.get('long') == '+' else False
            }
        return {'vowels': vowels, 'consonants': consonants}
    except Exception as e:
        print(f"Could not load phonological features: {e}")
        return {'vowels': {'a': {}, 'e': {}, 'i': {}, 'o': {}, 'u': {}}, 'consonants': {}}

def load_luganda_lexicon(dict_path="lg_lexicon.dict"):
    lexicon = {}
    try:
        with open(dict_path, 'r', encoding='utf-8') as f:
            for line in f:
                if '|' in line:
                    word, syllables = line.strip().split('|', 1)
                    lexicon[word.lower()] = syllables
        return lexicon
    except Exception as e:
        print(f"Could not load lexicon: {e}")
        return {}

# Load the resources
phono_features = load_phonological_features("allophones.lg.xml")
luganda_lexicon = load_luganda_lexicon("lg_lexicon.dict")

print(f"Loaded phonological features for {len(phono_features['vowels'])} vowels and {len(phono_features['consonants'])} consonants")
print(f"Loaded lexicon with {len(luganda_lexicon)} words")


Loaded phonological features for 10 vowels and 71 consonants
Loaded lexicon with 1843 words


In [ ]:
# Step 1: Set up your environment
!pip install sentencepiece transformers pandas numpy matplotlib seaborn tqdm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Step 4: Extract Morphological Patterns

def extract_morphological_patterns():
    # Noun class prefixes (from our analysis)
    noun_class_prefixes = {
        "1/2": ["mu", "ba"],
        "3/4": ["mu", "mi"],
        "5/6": ["li", "ma"],
        "7/8": ["ki", "bi"],
        "9/10": ["n", "n"],
        "11/10": ["lu", "n"],
        "12/14": ["ka", "bu"],
        "15": ["ku"],
        "16/17/18": ["wa", "ku", "mu"]
    }

    # Verbal extensions identified from analysis
    verbal_extensions = {
        "passive": ["wa", "ibwa"],
        "applied": ["era", "ira", "idira"],
        "stative": ["ka"],
        "repetitive": ["nga"],
        "causative": ["isa", "esa", "iza"],
        "reversive": ["ula", "ulula"],
        "intensive": ["irira"],
        "reciprocal": ["gana", "agana"]
    }

    # Known morpheme boundaries from the corpus
    morpheme_boundaries = {
        "prefix_stem": [
            "mu-ka", "n-ny", "n-na", "ku-ba", "ga-vu", "mu-li", "mu-ga",
            "ba-li", "mu-sa", "n-ge", "n-so", "bu-lu", "ku-ko"
        ],
        "stem_suffix": [
            "ib-wa", "eb-wa", "dd-wa", "wa-nga", "ee-ka", "ir-iza",
            "iz-ibwa", "si-nga", "ir-ira", "ba-nga", "ir-wa", "se-era"
        ]
    }

    # Geminates (doubled consonants) that should be preserved
    geminates = [
        "mm", "nn", "jj", "pp", "tt", "kk", "bb", "dd", "gg", "ff", "ss", "zz"
    ]

    # Common CV patterns that might be important for syllabification
    cv_patterns = [
        "CV", "CVCV", "CVCVCV", "VCVCV", "VCVCVCV", "CVCCV", "VCVCCV"
    ]

    return {
        "noun_class_prefixes": noun_class_prefixes,
        "verbal_extensions": verbal_extensions,
        "morpheme_boundaries": morpheme_boundaries,
        "geminates": geminates,
        "cv_patterns": cv_patterns
    }

# Extract morphological patterns
morpho_patterns = extract_morphological_patterns()

# Function to extract user-defined tokens for SentencePiece based on morphological patterns
def get_user_defined_tokens():
    tokens = []

    # Add noun class prefixes
    for class_group, prefixes in morpho_patterns["noun_class_prefixes"].items():
        tokens.extend(prefixes)

    # Add verbal extensions
    for ext_type, exts in morpho_patterns["verbal_extensions"].items():
        tokens.extend(exts)

    # Add some common prefixes and suffixes observed in the corpus
    tokens.extend(['o', 'e', 'a', 'b', 'k', 'ab', 'ok', 'oku', 'aba'])
    tokens.extend(['a', 'i', 'o', 'e', 'u', 'la', 'na', 'ra', 'ga'])

    # Add geminates
    tokens.extend(morpho_patterns["geminates"])

    # Ensure tokens are unique
    return list(set(tokens))

user_defined_tokens = get_user_defined_tokens()

# Print out the extracted patterns and token count for verification
print("Morphological patterns extracted:")
print("Noun class prefixes:", morpho_patterns["noun_class_prefixes"])
print("Verbal extensions:", morpho_patterns["verbal_extensions"])
print("Number of user-defined tokens for SentencePiece:", len(user_defined_tokens))


Morphological patterns extracted:
Noun class prefixes: {'1/2': ['mu', 'ba'], '3/4': ['mu', 'mi'], '5/6': ['li', 'ma'], '7/8': ['ki', 'bi'], '9/10': ['n', 'n'], '11/10': ['lu', 'n'], '12/14': ['ka', 'bu'], '15': ['ku'], '16/17/18': ['wa', 'ku', 'mu']}
Verbal extensions: {'passive': ['wa', 'ibwa'], 'applied': ['era', 'ira', 'idira'], 'stative': ['ka'], 'repetitive': ['nga'], 'causative': ['isa', 'esa', 'iza'], 'reversive': ['ula', 'ulula'], 'intensive': ['irira'], 'reciprocal': ['gana', 'agana']}
Number of user-defined tokens for SentencePiece: 53


In [ ]:
import re
import numpy as np

class EnhancedLugandaTokenizer:
    def __init__(self):
        """
        Comprehensive Luganda linguistic resource manager.
        Incorporates full noun class system, phonological features, tonal aspects,
        and morphological rules based on Luganda grammar.
        """
        # Complete noun class system with all possible prefixes
        self.linguistic_features = {
            'noun_classes': {
                'class_1': {
                    'singular_prefixes': ['omu', 'mu', 'omw', 'mw'],
                    'plural_prefixes': ['aba', 'ba', 'ab', 'b']
                },
                'class_2': {
                    'singular_prefixes': ['omu', 'mu', 'omw', 'mw'],
                    'plural_prefixes': ['emi', 'mi', 'emy', 'my']
                },
                'class_3': {
                    'singular_prefixes': ['eri', 'ri', 'e', 'li', 'ery', 'ly', 'ama', 'ma', 'am'],
                    'plural_prefixes': ['ama', 'ma', 'am', 'ga', 'ag']
                },
                'class_4': {
                    'singular_prefixes': ['eki', 'ki', 'eky', 'ky', 'ech'],
                    'plural_prefixes': ['ebi', 'bi', 'eby', 'by']
                },
                'class_5': {
                    'singular_prefixes': ['aka', 'ka', 'ak'],
                    'plural_prefixes': ['obu', 'bu', 'obw', 'bw']
                },
                'class_6': {
                    'singular_prefixes': ['olu', 'lu', 'olw', 'lw'],
                    'plural_prefixes': ['en', 'n', 'enn', 'em', 'emm', 'ez']
                },
                'class_7': {
                    'singular_prefixes': ['ogu', 'gu', 'ogw', 'gw'],
                    'plural_prefixes': ['aga', 'ga', 'ag', 'g']
                },
                'class_8': {
                    'singular_prefixes': ['oku', 'ku', 'okw', 'kw'],
                    'plural_prefixes': ['ama', 'ma', 'am']
                },
                'class_9': {
                    'singular_prefixes': ['otu', 'tu', 'otw', 'tw'],
                    'plural_prefixes': []  # No plurals in this class
                },
                'class_10': {  # Class with no prefixes, mostly loanwords and animals
                    'singular_prefixes': ['e', 'en', 'em', 'eny'],
                    'plural_prefixes': ['e', 'en', 'em', 'eny']
                }
            },

            # Expanded verb affixes covering major forms
            'verb_prefixes': [
                # Subject prefixes (person markers)
                'n', 'o', 'a', 'tu', 'mu', 'ba',

                # Tense markers
                'a', 'li', 'ku', 'na', 'naa', 'ki',

                # Object markers
                'n', 'ku', 'mu', 'tu', 'ba', 'ki', 'bi', 'ga', 'li', 'zi',

                # Common combinations
                'ya', 'wa', 'ga', 'ba', 'na', 'te', 'si',

                # Noun class subject markers as verb prefixes
                'ki', 'bi', 'li', 'ga', 'lu', 'tu', 'ka', 'bu', 'gu', 'zi', 'zi',

                # More complex combinations
                'teba', 'siba', 'aba', 'baa', 'gaa', 'yaa', 'taa', 'tee'
            ],

            # Complete list of verb extensions
            'verb_extensions': [
                # Basic extensions
                'wa', 'ibwa', 'iwa', 'ebwa', 'era', 'ira', 'esa', 'eka',
                'iza', 'eza', 'ulwa', 'uka', 'ula', 'uka', 'isa',

                # Complex and compound extensions
                'eddwa', 'iddwa', 'izibwa', 'erera', 'irira', 'isiza',
                'ikiriza', 'ukiriza', 'ulula', 'agana', 'aŋŋa', 'isana',

                # Additional extensions identified in corpus
                'dde', 'awo', 'anga', 'idira', 'ingira', 'zalira',

                # Passive, causative, applicative, and reciprocal extensions
                'nga', 'agana', 'aganya'
            ],

            # Negation markers
            'negation_prefixes': ['te', 't', 'si', 's', 'tee', 'ta'],

            # Tense-aspect markers
            'tense_markers': {
                'past': ['a', 'dde', 'zze', 'ye'],
                'present': ['a', 'e', 'o'],
                'future': ['naa', 'li', 'jja'],
                'ongoing': ['kyaa', 'nga']
            },

            # Common roots - expanded
            'common_roots': {
                'verbs': [
                    'kol', 'som', 'tambul', 'lya', 'nywa', 'gend', 'tuul',
                    'wulir', 'lab', 'leet', 'w', 'yit', 'twal', 'wandiik',
                    'yig', 'kuŋŋaan', 'beerag', 'yiger', 'yiger', 'bal', 'beez',
                    'lowooz', 'sanyuk', 'saab', 'sek', 'zin', 'sib', 'sogol',
                    'teek', 'teekawo', 'gab', 'gabirwa', 'enyig', 'eenyig', 'suul'
                ],
                'nouns': [
                    'ntu', 'wala', 'sajja', 'ana', 'maka', 'tebe', 'tabo',
                    'yizi', 'somero', 'kazi', 'gezi', 'kulu', 'naku',
                    'lungi', 'bi', 'yombo', 'fumbo', 'sawo', 'lwaliro',
                    'kyala', 'tima', 'wuka', 'sana', 'solo', 'banga', 'kalu',
                    'vubuka', 'kulemb', 'zadd', 'gavu', 'maama', 'abwe'
                ]
            },

            # Semi-vowel compounds identified in the text
            'semi_vowel_compounds': [
                'pw', 'bw', 'tw', 'dw', 'cw', 'jw', 'kw', 'gw', 'sw', 'zw',
                'mw', 'nw', 'nyw', 'ŋw', 'lw', 'rw', 'py', 'by', 'ty', 'dy',
                'ky', 'gy', 'my', 'ny', 'ly', 'ry'
            ],

            # Nasal compounds identified in the text
            'nasal_compounds': [
                'mp', 'mb', 'nt', 'nd', 'nc', 'nj', 'nk', 'ng', 'nf', 'nv', 'ns', 'nz'
            ],

            # Double consonants that can start words
            'initial_double_consonants': [
                'dd', 'jj', 'ff', 'mm', 'nn', 'ŋŋ', 'ss', 'gg', 'bb'
            ],

            # Loanwords that should be handled specially
            'loanwords': [
                'gavumenti', 'poliisi', 'sikaala', 'disitulikiti', 'pulayimale',
                'paalamenti', 'pulojekiti', 'bbayibuli', 'kompyuta', 'kalenda'
            ]
        }

        # Phonological rules derived from the provided text
        self.phonological_rules = {
            'vowel_lengthening': True,  # Long/double vowels change meaning
            'consonant_gemination': True,  # Double consonants change meaning
            'vowel_harmony': True,
            'vowel_hiatus_resolution': True,  # Final vowel assimilated by initial vowel
            'tone_patterns': ['level', 'falling']  # Two primary tone patterns
        }

    def is_vowel(self, char):
        """Return True if char is a vowel."""
        return char.lower() in 'aeiouāēīōū'  # Including long vowels

    def is_consonant(self, char):
        """Return True if char is a consonant."""
        return char.isalpha() and not self.is_vowel(char)

    def is_loanword(self, word):
        """Check if a word is a known loanword."""
        word_lower = word.lower()
        for loanword in self.linguistic_features['loanwords']:
            if loanword in word_lower or word_lower in loanword:
                return True
        return False

    def calculate_morphological_complexity(self, word):
        """
        Compute a complexity score for the word based on:
          - Matching known noun prefixes and class markers
          - Vowel–consonant transitions
          - Matching known verb extensions
          - Double consonants and vowels (geminate detection)
          - Semi-vowel compounds and nasal compounds
          - Positional factors
        """
        complexity = 0

        # Check if it's a loanword first
        if self.is_loanword(word):
            return 0.5  # Low complexity for loanwords

        # 1. Check all noun class prefixes
        for noun_class in self.linguistic_features['noun_classes'].values():
            for prefix_type in ['singular_prefixes', 'plural_prefixes']:
                for prefix in noun_class[prefix_type]:
                    if word.startswith(prefix):
                        complexity += len(prefix) * 2.5
                        break

        # 2. Check negation prefixes (high priority)
        for neg in self.linguistic_features['negation_prefixes']:
            if word.startswith(neg):
                complexity += len(neg) * 3  # Higher weight for negation
                break

        # 3. Check verb prefixes
        for prefix in self.linguistic_features['verb_prefixes']:
            if word.startswith(prefix):
                complexity += len(prefix) * 2
                break

        # 4. Check verb extensions at the end
        for ext in sorted(self.linguistic_features['verb_extensions'], key=len, reverse=True):
            if word.endswith(ext):
                complexity += len(ext) * 2
                break

        # 5. Add complexity for vowel-consonant transitions
        # This is important in Luganda syllable structure
        vowel_consonant_transitions = sum(
            1 for i in range(len(word)-1)
            if (self.is_vowel(word[i]) and self.is_consonant(word[i+1])) or
               (self.is_consonant(word[i]) and self.is_vowel(word[i+1]))
        )
        complexity += vowel_consonant_transitions

        # 6. Check for double consonants and vowels (geminates)
        geminates = sum(
            2 for i in range(len(word)-1)
            if word[i] == word[i+1]
        )
        complexity += geminates  # Stronger weight for geminates as they're meaningful

        # 7. Check for semi-vowel compounds
        for compound in self.linguistic_features['semi_vowel_compounds']:
            if compound in word:
                complexity += 2

        # 8. Check for nasal compounds
        for compound in self.linguistic_features['nasal_compounds']:
            if compound in word:
                complexity += 2

        # 9. Add a positional factor that favors word boundaries and midpoints
        positional_complexity = sum(
            abs(i - len(word)/2) for i in range(len(word))
        ) / len(word) if len(word) > 0 else 0
        complexity += positional_complexity

        # 10. Check for common roots
        for category in ['verbs', 'nouns']:
            for root in self.linguistic_features['common_roots'][category]:
                if root in word:
                    pos = word.find(root)
                    # Add complexity at root boundaries
                    if pos > 0:
                        complexity += 3  # Before root boundary
                    if pos + len(root) < len(word):
                        complexity += 3  # After root boundary

        return complexity

    def find_morphological_structure(self, word):
        """
        Identify morphological components in a word based on Luganda patterns.
        Returns a dictionary with morphological breakdown.
        """
        lw = word.lower()
        result = {'remainder': lw}

        # Handle loanwords as special cases
        if self.is_loanword(lw):
            result['is_loanword'] = True
            return result

        # Extract compound negation + verb prefixes (like "teba")
        for neg in sorted(self.linguistic_features['negation_prefixes'], key=len, reverse=True):
            if lw.startswith(neg) and len(lw) > len(neg):
                remainder = lw[len(neg):]
                # Check if what follows is a verb prefix
                for prefix in sorted(self.linguistic_features['verb_prefixes'], key=len, reverse=True):
                    if remainder.startswith(prefix) and len(remainder) > len(prefix):
                        result['negation_prefix'] = neg
                        result['verb_prefix'] = prefix
                        result['remainder'] = remainder[len(prefix):]
                        return result

                # Just negation prefix
                result['negation_prefix'] = neg
                result['remainder'] = remainder
                return result

        # 1. Check for noun class prefixes
        for noun_class, prefixes in self.linguistic_features['noun_classes'].items():
            for prefix_type in ['singular_prefixes', 'plural_prefixes']:
                for prefix in sorted(prefixes[prefix_type], key=len, reverse=True):
                    if lw.startswith(prefix) and len(lw) > len(prefix):
                        result['noun_class'] = noun_class
                        result['prefix'] = prefix
                        result['prefix_type'] = prefix_type
                        result['remainder'] = lw[len(prefix):]
                        return result

        # 2. Check for verb prefixes
        for prefix in sorted(self.linguistic_features['verb_prefixes'], key=len, reverse=True):
            if lw.startswith(prefix) and len(lw) > len(prefix):
                result['verb_prefix'] = prefix
                result['remainder'] = lw[len(prefix):]
                return result

        # 3. Check for verb extensions at the end
        for ext in sorted(self.linguistic_features['verb_extensions'], key=len, reverse=True):
            if lw.endswith(ext) and len(lw) > len(ext):
                result['verb_extension'] = ext
                result['remainder'] = lw[:-len(ext)]
                return result

        # 4. Check for common verb and noun roots
        for category in ['verbs', 'nouns']:
            for root in sorted(self.linguistic_features['common_roots'][category], key=len, reverse=True):
                if root in lw:
                    pos = lw.find(root)
                    # Only consider if the root is not the entire word
                    if len(lw) > len(root):
                        # Record both prefix and suffix if applicable
                        if pos > 0:
                            result['prefix'] = lw[:pos]
                        if pos + len(root) < len(lw):
                            result['suffix'] = lw[pos+len(root):]
                        result['root'] = root
                        result['root_type'] = category[:-1]  # 'verb' or 'noun'
                        result['remainder'] = ''  # Clear remainder as we've decomposed the word
                        return result

        # If no clear markers found, return just the remainder
        return result

    def adaptive_morphological_segmentation(self, word):
        """
        Apply Luganda-specific morphological segmentation using:
        1. Knowledge of noun class prefixes and verb affixes
        2. Rules for consonant gemination and vowel lengthening
        3. Statistical complexity to identify morpheme boundaries
        4. Patterns of semi-vowel and nasal compounds
        5. Special handling for loanwords
        """
        if len(word) <= 3:
            return word  # Too short to segment

        # Handle capitalization
        original_word = word
        lw = word.lower()

        # Special handling for loanwords
        if self.is_loanword(lw):
            return original_word

        # Primary morphological analysis
        structure = self.find_morphological_structure(lw)
        segments = []

        # Handle negation prefix if found (highest priority)
        if 'negation_prefix' in structure:
            segments.append(structure['negation_prefix'])
            if 'verb_prefix' in structure:
                segments.append(structure['verb_prefix'])
                remainder = structure.get('remainder', '')
            else:
                remainder = structure.get('remainder', lw)
        # Handle prefixes if found
        elif 'prefix' in structure:
            segments.append(structure['prefix'])
            remainder = structure.get('remainder', lw)
        else:
            remainder = structure.get('remainder', lw)

        # Handle roots, extensions, and suffixes
        if 'root' in structure:
            segments.append(structure['root'])
            if 'suffix' in structure:
                segments.append(structure['suffix'])
            remainder = ''
        elif 'verb_extension' in structure:
            segments.append(structure.get('remainder', ''))
            segments.append(structure['verb_extension'])
            remainder = ''

        # Complexity-based segmentation for remaining pieces
        if remainder:
            if len(remainder) > 4:  # Only attempt if long enough
                # Calculate complexity scores for potential split points
                scores = [self.calculate_morphological_complexity(remainder[:i+1])
                         for i in range(1, len(remainder))]

                if scores:  # Check if scores list is not empty
                    # Use a higher threshold for Luganda (80th percentile)
                    threshold = np.percentile(scores, 80)
                    boundaries = [i for i, s in enumerate(scores) if s >= threshold]

                    if boundaries:
                        segs = []
                        last = 0
                        for b in boundaries:
                            if b - last > 1:  # Avoid tiny segments
                                segs.append(remainder[last:b+1])
                                last = b+1
                        if last < len(remainder):
                            segs.append(remainder[last:])
                        segments.append('·'.join(segs))
                    else:
                        segments.append(remainder)
                else:
                    segments.append(remainder)
            else:
                segments.append(remainder)

        # Join segments with the morpheme boundary marker
        result = '·'.join([seg for seg in segments if seg])

        # Preserve original capitalization
        if original_word[0].isupper() and result:
            result = result[0].upper() + result[1:]

        return result

    def tokenize(self, text):
        """
        Tokenize Luganda text with awareness of punctuation and sentence structure.
        """
        # Handle punctuation by separating it
        text = re.sub(r'([.,!?;:])', r' \1 ', text)

        # Split into words
        words = text.split()
        processed_words = []

        for word in words:
            # Skip punctuation
            if word in ".,!?;:":
                processed_words.append(word)
            else:
                processed_words.append(self.adaptive_morphological_segmentation(word))

        # Rejoin with spaces
        return " ".join(processed_words)

    def analyze_word(self, word):
        """
        Provide detailed linguistic analysis of a single word.
        """
        structure = self.find_morphological_structure(word.lower())
        segmented = self.adaptive_morphological_segmentation(word)

        # Check if it's a loanword
        is_loanword = self.is_loanword(word.lower())

        # Identify likely noun class
        noun_class = structure.get('noun_class', None)
        if not noun_class:
            for cls, prefixes in self.linguistic_features['noun_classes'].items():
                for prefix in prefixes['singular_prefixes'] + prefixes['plural_prefixes']:
                    if word.lower().startswith(prefix):
                        noun_class = cls
                        break
                if noun_class:
                    break

        # Check for verbal features
        is_verb = False
        has_negation = 'negation_prefix' in structure
        has_verb_extension = False

        # Check for verb extension
        for ext in self.linguistic_features['verb_extensions']:
            if word.lower().endswith(ext):
                has_verb_extension = True
                break

        # Check if likely a verb
        if has_verb_extension or 'verb_prefix' in structure:
            is_verb = True
        elif 'root_type' in structure and structure['root_type'] == 'verb':
            is_verb = True
        else:
            for prefix in self.linguistic_features['verb_prefixes']:
                if word.lower().startswith(prefix):
                    is_verb = True
                    break

        # Calculate confidence score (0-100%)
        confidence = 100
        if is_loanword:
            confidence = 90  # High confidence for identified loanwords
        elif len(segmented.split('·')) < 2:
            confidence = 60  # Lower confidence if no segmentation occurred
        elif not noun_class and not is_verb:
            confidence = 70  # Lower confidence if we couldn't identify word class

        # Calculate morpheme count
        morpheme_count = len(segmented.split('·'))

        return {
            "word": word,
            "segmented": segmented,
            "morphological_structure": structure,
            "is_loanword": is_loanword,
            "likely_noun_class": noun_class,
            "likely_verb": is_verb,
            "has_verb_extension": has_verb_extension,
            "has_negation": has_negation,
            "morpheme_count": morpheme_count,
            "confidence_score": confidence
        }


In [ ]:
def test_enhanced_tokenizer():
    """
    Comprehensive test of the enhanced Luganda tokenizer on various example types.
    """
    tokenizer = EnhancedLugandaTokenizer()

    print("ENHANCED LUGANDA TOKENIZER - COMPREHENSIVE TEST")
    print("=" * 70)

    # Test Categories

    # 1. Basic sentences with different grammatical structures
    basic_sentences = [
        "Kansala yaweereddwa akakalu ka poliisi.",
        "Abazadde tebalina sente zimala kutwala baana baabwe ku ssomero.",
        "Amasomero gateekeddwa okugabira abayizi abagezi bbasale ne sikaala."
    ]

    # 2. Complex verbs with multiple affixes
    complex_verbs = [
        "yaweereddwa", "tebalina", "bateekeddwa", "beenyigidde", "baasuulawo",
        "abateesobola", "gaateekeddwa", "okujjumbira", "tekwetaagisa", "tekinnategeerekeka"
    ]

    # 3. Nouns with class markers
    class_nouns = [
        "abantu", "omuntu", "emiti", "omuti", "amasomero", "essomero",
        "ebitabo", "ekitabo", "obukalu", "akakalu", "ennyumba", "oluuyi"
    ]

    # 4. Loanwords
    loanwords = [
        "gavumenti", "poliisi", "disitulikiti", "pulayimale", "sikaala",
        "paalamenti", "pulojekiti", "kalenda", "bbayibuli", "kompyuta"
    ]

    # 5. Words with negation
    negation_words = [
        "tebalina", "tebagabirira", "tebasobola", "tebaalina",
        "tebamalaayo", "tekwetaagisa", "tekinnategeerekeka"
    ]

    # Testing each category
    print("\n1. BASIC SENTENCES")
    print("-" * 50)
    for i, sentence in enumerate(basic_sentences, 1):
        tokenized = tokenizer.tokenize(sentence)
        print(f"\nSentence {i}:")
        print(f"Original: {sentence}")
        print(f"Tokenized: {tokenized}")

    print("\n\n2. COMPLEX VERBS")
    print("-" * 50)
    print("Word".ljust(25) + "Segmentation".ljust(30) + "Analysis")
    print("-" * 70)
    for word in complex_verbs:
        analysis = tokenizer.analyze_word(word)
        segmented = analysis["segmented"]
        noun_class = analysis["likely_noun_class"] or "N/A"
        is_verb = "Yes" if analysis["likely_verb"] else "No"
        has_extension = "Yes" if analysis["has_verb_extension"] else "No"
        has_negation = "Yes" if analysis["has_negation"] else "No"
        confidence = f"{analysis['confidence_score']}%"

        print(f"{word.ljust(25)}{segmented.ljust(30)}Verb: {is_verb}, Ext: {has_extension}, Neg: {has_negation}, Conf: {confidence}")

    print("\n\n3. NOUN CLASS EXAMPLES")
    print("-" * 50)
    print("Word".ljust(15) + "Segmentation".ljust(20) + "Noun Class".ljust(15) + "Confidence")
    print("-" * 70)
    for word in class_nouns:
        analysis = tokenizer.analyze_word(word)
        segmented = analysis["segmented"]
        noun_class = analysis["likely_noun_class"] or "N/A"
        confidence = f"{analysis['confidence_score']}%"

        print(f"{word.ljust(15)}{segmented.ljust(20)}{str(noun_class).ljust(15)}{confidence}")

    print("\n\n4. LOANWORD HANDLING")
    print("-" * 50)
    print("Word".ljust(15) + "Is Loanword".ljust(15) + "Segmentation")
    print("-" * 70)
    for word in loanwords:
        analysis = tokenizer.analyze_word(word)
        is_loanword = "Yes" if analysis["is_loanword"] else "No"
        segmented = analysis["segmented"]

        print(f"{word.ljust(15)}{is_loanword.ljust(15)}{segmented}")

    print("\n\n5. NEGATION HANDLING")
    print("-" * 50)
    print("Word".ljust(20) + "Segmentation".ljust(25) + "Has Negation".ljust(15) + "Morpheme Count")
    print("-" * 70)
    for word in negation_words:
        analysis = tokenizer.analyze_word(word)
        segmented = analysis["segmented"]
        has_negation = "Yes" if analysis["has_negation"] else "No"
        morpheme_count = analysis["morpheme_count"]

        print(f"{word.ljust(20)}{segmented.ljust(25)}{has_negation.ljust(15)}{morpheme_count}")

    # Overall statistics
    all_words = basic_sentences + complex_verbs + class_nouns + loanwords + negation_words
    flattened_words = []
    for item in all_words:
        if " " in item:  # It's a sentence
            flattened_words.extend([w for w in item.split() if w not in ".,!?;:"])
        else:
            flattened_words.append(item)

    unique_words = set(flattened_words)

    print(f"\n\nSUMMARY STATISTICS")
    print("=" * 50)
    print(f"Total unique words tested: {len(unique_words)}")

    # Calculate average segmentations per word
    segmentation_counts = []
    for word in unique_words:
        analysis = tokenizer.analyze_word(word)
        seg_count = len(analysis["segmented"].split("·"))
        segmentation_counts.append(seg_count)

    avg_segments = sum(segmentation_counts) / len(segmentation_counts)
    print(f"Average morphemes per word: {avg_segments:.2f}")

    # Calculate confidence distribution
    confidence_levels = {"90-100%": 0, "80-89%": 0, "70-79%": 0, "60-69%": 0, "Below 60%": 0}
    for word in unique_words:
        analysis = tokenizer.analyze_word(word)
        confidence = analysis["confidence_score"]

        if confidence >= 90:
            confidence_levels["90-100%"] += 1
        elif confidence >= 80:
            confidence_levels["80-89%"] += 1
        elif confidence >= 70:
            confidence_levels["70-79%"] += 1
        elif confidence >= 60:
            confidence_levels["60-69%"] += 1
        else:
            confidence_levels["Below 60%"] += 1

    print("\nConfidence Distribution:")
    for level, count in confidence_levels.items():
        percentage = (count / len(unique_words)) * 100
        print(f"  {level}: {count} words ({percentage:.1f}%)")

    return tokenizer

# Run the test
test_enhanced_tokenizer()

ENHANCED LUGANDA TOKENIZER - COMPREHENSIVE TEST

1. BASIC SENTENCES
--------------------------------------------------

Sentence 1:
Original: Kansala yaweereddwa akakalu ka poliisi.
Tokenized: Ka·nsal·a weeredd·wa aka·kalu ka poliisi .

Sentence 2:
Original: Abazadde tebalina sente zimala kutwala baana baabwe ku ssomero.
Tokenized: Aba·zadd·e te·ba·lina s·ente mala ku·twal·a ba·ana ba·abwe ku s·somer·o .

Sentence 3:
Original: Amasomero gateekeddwa okugabira abayizi abagezi bbasale ne sikaala.
Tokenized: Ama·somer·o ga·teekedd·wa oku·gabir·a aba·yizi aba·gezi b·basal·e ne sikaala .


2. COMPLEX VERBS
--------------------------------------------------
Word                     Segmentation                  Analysis
----------------------------------------------------------------------
yaweereddwa              weeredd·wa                    Verb: Yes, Ext: Yes, Neg: No, Conf: 100%
tebalina                 te·ba·lina                    Verb: Yes, Ext: No, Neg: Yes, Conf: 100%
bateekeddwa   

In [ ]:
import sentencepiece as spm
import os
import re
import random
import time
from tqdm import tqdm

class MorphologicalSentencePiece:
    def __init__(self, luganda_tokenizer, corpus_path=None):
        """
        Initialize the morphologically-aware SentencePiece tokenizer for Luganda.

        Args:
            luganda_tokenizer: Instance of EnhancedLugandaTokenizer.
            corpus_path: Path to the Luganda corpus file (optional).
        """
        self.luganda_tokenizer = luganda_tokenizer
        self.corpus_path = corpus_path
        self.sp_model = None
        self.model_path = None

    def preprocess_corpus(self, output_file="luganda_preprocessed.txt", sample_size=None):
        """
        Preprocess the Luganda corpus with morphological segmentation.

        Args:
            output_file: Path to save the preprocessed corpus.
            sample_size: Number of sentences to sample (None for all).

        Returns:
            Path to the preprocessed corpus.
        """
        if not self.corpus_path:
            raise ValueError("Corpus path not specified. Set corpus_path first.")

        print(f"Loading corpus from {self.corpus_path}...")
        with open(self.corpus_path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f if line.strip()]

        if sample_size is not None and sample_size < len(lines):
            print(f"Sampling {sample_size} sentences from corpus of {len(lines)} sentences...")
            lines = random.sample(lines, sample_size)

        total_lines = len(lines)
        print(f"Preprocessing {total_lines} sentences with morphological analysis...")

        with open(output_file, 'w', encoding='utf-8') as f:
            for line in tqdm(lines):
                segmented = self.luganda_tokenizer.tokenize(line)
                preprocessed = segmented.replace("·", " ")
                f.write(preprocessed + '\n')

        print(f"Preprocessed corpus saved to {output_file}")
        return output_file

    def extract_user_defined_symbols(self):
        """Extract linguistic units as user-defined symbols for SentencePiece."""
        symbols = []
        # Add noun class prefixes
        for class_info in self.luganda_tokenizer.linguistic_features['noun_classes'].values():
            for prefix_list in [class_info['singular_prefixes'], class_info['plural_prefixes']]:
                for prefix in prefix_list:
                    if len(prefix) > 1:
                        symbols.append(prefix)
        # Add verb extensions
        symbols.extend([ext for ext in self.luganda_tokenizer.linguistic_features['verb_extensions'] if len(ext) > 1])
        # Add negation prefixes
        symbols.extend(self.luganda_tokenizer.linguistic_features['negation_prefixes'])
        # Add common verb prefixes
        symbols.extend([prefix for prefix in self.luganda_tokenizer.linguistic_features['verb_prefixes'] if len(prefix) > 1])
        return list(set([sym for sym in symbols if 2 <= len(sym) <= 8]))

    def train(self, input_file=None, model_prefix="luganda_spm",
              vocab_size=10000, character_coverage=0.9995):
        """
        Train a SentencePiece model with Luganda morphological awareness.

        Args:
            input_file: Path to the preprocessed corpus (if None, uses preprocessed file).
            model_prefix: Prefix for the SentencePiece model files.
            vocab_size: Size of the vocabulary (default: 10,000).
            character_coverage: Character coverage ratio.

        Returns:
            Path to the trained SentencePiece model.
        """
        if input_file is None:
            input_file = "luganda_preprocessed.txt"
            if not os.path.exists(input_file):
                input_file = self.preprocess_corpus(output_file=input_file)

        user_defined_symbols = self.extract_user_defined_symbols()
        print(f"Using {len(user_defined_symbols)} morphologically significant tokens as user-defined symbols")

        train_args = {
            'input': input_file,
            'model_prefix': model_prefix,
            'vocab_size': vocab_size,
            'character_coverage': character_coverage,
            'model_type': 'unigram',
            'user_defined_symbols': user_defined_symbols,
            'split_digits': False,
            'byte_fallback': True,
            'allow_whitespace_only_pieces': False,
            'unk_surface': '<UNK>',
            'normalization_rule_name': 'identity',
            'max_sentencepiece_length': 16,
            'num_threads': 8
        }

        print(f"Training SentencePiece model with vocabulary size {vocab_size}...")
        start_time = time.time()
        try:
            spm.SentencePieceTrainer.train(**train_args)
        except RuntimeError as e:
            error_msg = str(e)
            if "Vocabulary size too high" in error_msg:
                max_vocab_match = re.search(r'<= (\d+)', error_msg)
                if max_vocab_match:
                    max_vocab = int(max_vocab_match.group(1))
                    print(f"Requested vocabulary size {vocab_size} is too large. Reducing to {max_vocab}")
                    train_args['vocab_size'] = max_vocab
                    spm.SentencePieceTrainer.train(**train_args)
                else:
                    raise
            else:
                raise
        training_time = time.time() - start_time
        print(f"Model training completed in {training_time:.2f} seconds")
        print(f"Model saved as {model_prefix}.model and {model_prefix}.vocab")
        self.model_path = f"{model_prefix}.model"
        self.sp_model = spm.SentencePieceProcessor()
        self.sp_model.load(self.model_path)
        print(f"Loaded model with {self.sp_model.get_piece_size()} tokens")
        return self.model_path

    def tokenize(self, text, use_morphology=True):
        """
        Tokenize text using the trained SentencePiece model.

        Args:
            text: Input text to tokenize.
            use_morphology: Whether to apply morphological preprocessing.

        Returns:
            List of tokens.
        """
        if self.sp_model is None:
            raise ValueError("SentencePiece model not loaded. Train or load a model first.")
        if use_morphology:
            preprocessed = self.luganda_tokenizer.tokenize(text).replace("·", " ")
            return self.sp_model.encode_as_pieces(preprocessed)
        else:
            return self.sp_model.encode_as_pieces(text)

    def test_tokenization(self, test_sentences):
        """
        Test and display tokenization results on sample sentences.

        Args:
            test_sentences: List of sentences to test.

        Returns:
            Dictionary with tokenization statistics.
        """
        if self.sp_model is None:
            raise ValueError("SentencePiece model not loaded. Train or load a model first.")
        results = {
            'standard_tokens_per_sentence': [],
            'morpho_tokens_per_sentence': [],
            'morpheme_counts': []
        }
        print("\nTokenization Test:\n" + "-" * 50)
        for i, sentence in enumerate(test_sentences):
            print(f"\nSentence {i+1}: {sentence}")
            standard_tokens = self.sp_model.encode_as_pieces(sentence)
            print(f"Standard: {' '.join(standard_tokens)}")
            results['standard_tokens_per_sentence'].append(len(standard_tokens))
            morpho_processed = self.luganda_tokenizer.tokenize(sentence).replace("·", " ")
            morpho_tokens = self.sp_model.encode_as_pieces(morpho_processed)
            print(f"Morpho-aware: {' '.join(morpho_tokens)}")
            results['morpho_tokens_per_sentence'].append(len(morpho_tokens))
            original_morphemes = self.luganda_tokenizer.tokenize(sentence).count("·") + 1
            results['morpheme_counts'].append(original_morphemes)
            print(f"Tokens: Standard={len(standard_tokens)}, Morpho={len(morpho_tokens)}, Morphemes={original_morphemes}")
        results['avg_standard_tokens'] = sum(results['standard_tokens_per_sentence']) / len(test_sentences)
        results['avg_morpho_tokens'] = sum(results['morpho_tokens_per_sentence']) / len(test_sentences)
        results['avg_morphemes'] = sum(results['morpheme_counts']) / len(test_sentences)
        results['morpho_efficiency'] = results['avg_morpho_tokens'] / results['avg_morphemes']
        print("\nSummary Statistics:")
        print(f"Average tokens per sentence (standard): {results['avg_standard_tokens']:.2f}")
        print(f"Average tokens per sentence (morpho-aware): {results['avg_morpho_tokens']:.2f}")
        print(f"Average morphemes per sentence: {results['avg_morphemes']:.2f}")
        print(f"Morphological efficiency (tokens/morpheme): {results['morpho_efficiency']:.2f}")
        return results


In [ ]:
# Next cell: Train the SentencePiece model and test tokenization

# Initialize the EnhancedLugandaTokenizer (make sure its definition is in a previous cell)
tokenizer = EnhancedLugandaTokenizer()

# Set the path to your corpus file (ensure "MergedTok.txt" is in your working directory)
corpus_path = "MergedTok.txt"

# Create the MorphologicalSentencePiece processor using your tokenizer and corpus path
morph_sp = MorphologicalSentencePiece(tokenizer, corpus_path=corpus_path)

# Preprocess the corpus (using 50,000 sentences; remove sample_size to process the entire corpus)
preprocessed_file = morph_sp.preprocess_corpus(sample_size=50000)

# Train the SentencePiece model (vocab size 10,000; will auto-adjust if needed)
model_path = morph_sp.train(input_file=preprocessed_file, vocab_size=10000)

# Test tokenization on a set of sample sentences
test_sentences = [
    "Kansala yaweereddwa akakalu ka poliisi.",
    "Abazadde tebalina sente zimala kutwala baana baabwe ku ssomero.",
    "Amasomero gateekeddwa okugabira abayizi abagezi bbasale ne sikaala.",
    "Omulimu gwo gukoleddwa bulungi nnyo."
]
tokenization_stats = morph_sp.test_tokenization(test_sentences)

print("\nModel saved to:", model_path)


Loading corpus from MergedTok.txt...
Sampling 50000 sentences from corpus of 108765 sentences...
Preprocessing 50000 sentences with morphological analysis...


100%|██████████| 50000/50000 [01:03<00:00, 793.12it/s]
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: luganda_preprocessed.txt
  input_format: 
  model_prefix: luganda_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 8
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: emi
  user_defined_symbols: enn
  user_defined_symbols: eri
  user_defined_symbols: eky
  user_defined_symbols: ebi
  user_defined_symbols: si
  user_defined_symbols: awo
  user_defined_symbols: aka
  user_defined_symbols: ama
  user_defined_symbols: iwa
  user_de

Preprocessed corpus saved to luganda_preprocessed.txt
Using 108 morphologically significant tokens as user-defined symbols
Training SentencePiece model with vocabulary size 10000...
Requested vocabulary size 10000 is too large. Reducing to 5795
Model training completed in 2.13 seconds
Model saved as luganda_spm.model and luganda_spm.vocab
Loaded model with 5795 tokens

Tokenization Test:
--------------------------------------------------

Sentence 1: Kansala yaweereddwa akakalu ka poliisi.
Standard: ▁Ka n sala ▁ ya weer eddwa ▁ ak aka lu ▁ ka ▁po li i si .
Morpho-aware: ▁Ka ▁nsal ▁a ▁weeredd ▁ wa ▁ aka ▁ ka lu ▁ ka ▁po li i si ▁.
Tokens: Standard=18, Morpho=18, Morphemes=5

Sentence 2: Abazadde tebalina sente zimala kutwala baana baabwe ku ssomero.
Standard: ▁A ba za dde ▁ teba li na ▁s en te ▁ zi ma la ▁ ku tw ala ▁ baa na ▁ baa bw e ▁ ku ▁ ssome ro .
Morpho-aware: ▁A ba ▁zadd ▁e ▁ te ▁ ba ▁ li na ▁s ▁ en te ▁ ma la ▁ ku ▁ tw al ▁a ▁ ba ▁a na ▁ ba ▁ ab we ▁ ku ▁s ▁somer ▁o ▁.
Tokens: 

 ez
  user_defined_symbols: aba
  user_defined_symbols: emy
  user_defined_symbols: taa
  user_defined_symbols: isiza
  user_defined_symbols: baa
  user_defined_symbols: omu
  user_defined_symbols: ak
  user_defined_symbols: teba
  user_defined_symbols: mu
  user_defined_symbols: emm
  user_defined_symbols: isa
  user_defined_symbols: esa
  user_defined_symbols: ka
  user_defined_symbols: oku
  user_defined_symbols: tu
  user_defined_symbols: olw
  user_defined_symbols: ebwa
  user_defined_symbols: eddwa
  user_defined_symbols: eny
  user_defined_symbols: ba
  user_defined_symbols: nga
  user_defined_symbols: aga
  user_defined_symbols: my
  user_defined_symbols: ogw
  user_defined_symbols: ab
  user_defined_symbols: ku
  user_defined_symbols: ri
  user_defined_symbols: zalira
  user_defined_symbols: aganya
  user_defined_symbols: agana
  user_defined_symbols: okw
  user_defined_symbols: ibwa
  user_defined_symbols: ogu
  user_defined_symbols: erera
  user_defined_symbols: en
  user_de

In [ ]:
!pip install sentencepiece


[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load("luganda_spm.model")

True

In [ ]:
tokenizer = EnhancedLugandaTokenizer()

In [ ]:
# Define a test sentence
test_sentence = "Abantu balina okuyiga oluganda."

# Tokenize using SentencePiece directly
tokens = sp.encode_as_pieces(test_sentence)
print("Standard tokenization:")
print(tokens)
print(f"Number of tokens: {len(tokens)}")

# To use your full morphological tokenizer, you'll need to reinitialize your EnhancedLugandaTokenizer
# For example:
# tokenizer = EnhancedLugandaTokenizer()
# morphed_text = tokenizer.tokenize(test_sentence)
# print("\nMorphological tokenization:")
# print(morphed_text)

Standard tokenization:
['▁A', 'ba', 'n', 'tu', '▁', 'ba', 'li', 'na', '▁', 'oku', 'yi', 'ga', '▁', 'olu', 'ga', 'nda', '.']
Number of tokens: 17


In [ ]:
# Initialize the enhanced tokenizer
tokenizer = EnhancedLugandaTokenizer()

# Test sentence
test_sentence = "Abantu balina okuyiga oluganda."

# Apply morphological segmentation
morphed_text = tokenizer.tokenize(test_sentence)
print("Morphologically segmented text:")
print(morphed_text)

# Replace the · with spaces for SentencePiece processing
morph_for_sp = morphed_text.replace("·", " ")
print("\nText prepared for SentencePiece:")
print(morph_for_sp)

# Apply SentencePiece on the morphologically segmented text
tokens = sp.encode_as_pieces(morph_for_sp)
print("\nFully morphologically-aware tokenization:")
print(tokens)
print(f"Number of tokens: {len(tokens)}")

# Compare with standard tokenization
standard_tokens = sp.encode_as_pieces(test_sentence)
print("\nStandard tokenization (for comparison):")
print(standard_tokens)
print(f"Number of tokens: {len(standard_tokens)}")

Morphologically segmented text:
Aba·ntu ba·lina oku·yiga olu·gand·a .

Text prepared for SentencePiece:
Aba ntu ba lina oku yiga olu gand a .

Fully morphologically-aware tokenization:
['▁A', 'ba', '▁n', 'tu', '▁', 'ba', '▁', 'li', 'na', '▁', 'oku', '▁yi', 'ga', '▁', 'olu', '▁', 'ga', 'nd', '▁a', '▁.']
Number of tokens: 20

Standard tokenization (for comparison):
['▁A', 'ba', 'n', 'tu', '▁', 'ba', 'li', 'na', '▁', 'oku', 'yi', 'ga', '▁', 'olu', 'ga', 'nda', '.']
Number of tokens: 17


In [ ]:
import sentencepiece as spm
import pickle

class ABLT:
    """
    Alkebulan Bantu-Luganda Tokenizer (ABLT)

    A unified tokenizer that combines the EnhancedLugandaTokenizer's morphological
    analysis with SentencePiece tokenization optimized for Luganda.
    """

    def __init__(self, model_path="luganda_spm.model"):
        """
        Initialize the ABLT tokenizer.

        Args:
            model_path (str): Path to the trained SentencePiece model file
        """
        # Store the model path
        self.model_path = model_path

        # Initialize SentencePiece processor
        self.sp_tokenizer = spm.SentencePieceProcessor()
        loaded = self.sp_tokenizer.load(model_path)
        if not loaded:
            raise ValueError(f"Failed to load SentencePiece model from {model_path}")

        # Initialize the EnhancedLugandaTokenizer (your existing code)
        self.morphological_tokenizer = EnhancedLugandaTokenizer()

    def tokenize_text(self, text):
        """
        Apply full tokenization pipeline (morphological + SentencePiece).

        Args:
            text (str): Input text to tokenize

        Returns:
            list: List of tokens
        """
        # Step 1: Apply morphological segmentation
        morphed_text = self.morphological_tokenizer.tokenize(text)

        # Step 2: Replace morpheme boundaries with spaces for SentencePiece
        prepped_text = morphed_text.replace("·", " ")

        # Step 3: Apply SentencePiece tokenization
        tokens = self.sp_tokenizer.encode_as_pieces(prepped_text)

        return tokens

    def encode(self, text):
        """
        Convert text to token IDs.

        Args:
            text (str): Input text to encode

        Returns:
            list: List of token IDs
        """
        # Apply morphological segmentation
        morphed_text = self.morphological_tokenizer.tokenize(text)

        # Replace morpheme boundaries with spaces
        prepped_text = morphed_text.replace("·", " ")

        # Convert to token IDs
        token_ids = self.sp_tokenizer.encode(prepped_text)

        return token_ids

    def decode(self, token_ids):
        """
        Convert token IDs back to text.

        Args:
            token_ids (list): List of token IDs

        Returns:
            str: Decoded text
        """
        return self.sp_tokenizer.decode(token_ids)

    def get_tokens_with_importance(self, text):
        """
        Get tokens with importance scores for Simpson-based attention.

        Args:
            text (str): Input text

        Returns:
            tuple: (tokens, importance_scores)
        """
        # Step 1: Apply morphological segmentation
        morphed_text = self.morphological_tokenizer.tokenize(text)
        morphemes = morphed_text.replace("·", " ").split()

        # Step 2: Calculate importance scores for morphemes
        importance_map = {}

        for morpheme in morphemes:
            # Calculate importance based on morphological role
            importance = 0

            # Check if it's a prefix
            for noun_class in self.morphological_tokenizer.linguistic_features['noun_classes'].values():
                for prefix_type in ['singular_prefixes', 'plural_prefixes']:
                    for prefix in noun_class[prefix_type]:
                        if morpheme == prefix:
                            importance += 2.0  # Noun class prefixes are important

            # Check if it's a verb prefix
            if morpheme in self.morphological_tokenizer.linguistic_features['verb_prefixes']:
                importance += 2.5  # Verb prefixes are very important

            # Check if it's a verb extension
            if morpheme in self.morphological_tokenizer.linguistic_features['verb_extensions']:
                importance += 1.5  # Verb extensions are moderately important

            # Check if it's a negation prefix
            if morpheme in self.morphological_tokenizer.linguistic_features['negation_prefixes']:
                importance += 3.0  # Negation is highly important

            # If no specific importance found, use length as a proxy
            if importance == 0:
                importance = min(1.0, len(morpheme) / 5.0)

            importance_map[morpheme] = importance

        # Step 3: Get SentencePiece tokens
        tokens = self.sp_tokenizer.encode_as_pieces(morphed_text.replace("·", " "))

        # Step 4: Assign importance scores to tokens
        importance_scores = []
        for token in tokens:
            # Remove the leading space marker if present
            clean_token = token.replace('▁', '')

            # Find the closest morpheme for this token
            if clean_token in importance_map:
                importance_scores.append(importance_map[clean_token])
            else:
                # For subword tokens, use a lower score
                importance_scores.append(0.5)

        return tokens, importance_scores

    def save(self, path):
        """
        Save the ABLT tokenizer to disk.

        Args:
            path (str): Path to save the tokenizer
        """
        # Store the model path
        data = {
            'model_path': self.model_path
        }

        with open(path, 'wb') as f:
            pickle.dump(data, f)

    @classmethod
    def load(cls, path):
        """
        Load an ABLT tokenizer from disk.

        Args:
            path (str): Path to the saved tokenizer

        Returns:
            ABLT: The loaded tokenizer
        """
        with open(path, 'rb') as f:
            data = pickle.load(f)

        # Create a new instance
        tokenizer = cls(model_path=data['model_path'])
        return tokenizer

In [ ]:
# Initialize the ABLT tokenizer (do this only once)
ablt = ABLT("luganda_spm.model")

# Example usage
text = "Abantu balina okuyiga oluganda."

# Basic tokenization
tokens = ablt.tokenize_text(text)  # Changed to tokenize_text
print("Tokens:", tokens)

# Get tokens with importance scores for Simpson-based attention
tokens, importance = ablt.get_tokens_with_importance(text)
print("\nTokens with importance:")
for t, i in zip(tokens, importance):
    print(f"  {t}: {i:.2f}")

# Save the tokenizer for later use
ablt.save("ablt_tokenizer.pkl")

# Later, you can load it with just one line:
# ablt = ABLT.load("ablt_tokenizer.pkl")

Tokens: ['▁A', 'ba', '▁n', 'tu', '▁', 'ba', '▁', 'li', 'na', '▁', 'oku', '▁yi', 'ga', '▁', 'olu', '▁', 'ga', 'nd', '▁a', '▁.']

Tokens with importance:
  ▁A: 0.50
  ba: 4.50
  ▁n: 0.50
  tu: 0.50
  ▁: 0.50
  ba: 4.50
  ▁: 0.50
  li: 0.50
  na: 0.50
  ▁: 0.50
  oku: 2.00
  ▁yi: 0.50
  ga: 0.50
  ▁: 0.50
  olu: 2.00
  ▁: 0.50
  ga: 0.50
  nd: 0.50
  ▁a: 2.50
  ▁.: 0.20


In [ ]:
# Load your saved tokenizer
ablt = ABLT.load("ablt_tokenizer.pkl")

# Test basic tokenization
text = "Abantu balina okuyiga oluganda."
tokens = ablt.tokenize_text(text)
token_ids = ablt.encode(text)
tokens_with_importance = ablt.get_tokens_with_importance(text)

print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Tokens with Importance:", tokens_with_importance)

Tokens: ['▁A', 'ba', '▁n', 'tu', '▁', 'ba', '▁', 'li', 'na', '▁', 'oku', '▁yi', 'ga', '▁', 'olu', '▁', 'ga', 'nd', '▁a', '▁.']
Token IDs: [385, 70, 371, 65, 367, 70, 367, 49, 47, 367, 64, 433, 105, 367, 39, 367, 105, 405, 368, 369]
Tokens with Importance: (['▁A', 'ba', '▁n', 'tu', '▁', 'ba', '▁', 'li', 'na', '▁', 'oku', '▁yi', 'ga', '▁', 'olu', '▁', 'ga', 'nd', '▁a', '▁.'], [0.5, 4.5, 0.5, 0.5, 0.5, 4.5, 0.5, 0.5, 0.5, 0.5, 2.0, 0.5, 0.5, 0.5, 2.0, 0.5, 0.5, 0.5, 2.5, 0.2])


In [ ]:
import torch
import torch.nn as nn
import math

class SimpsonOptimizer(torch.optim.Optimizer):
    def __init__(
        self,
        params,
        lr=0.001,
        betas=(0.9, 0.999),
        history_size=5,  # Increased for A6000's memory
        eps=1e-8,
        weight_decay=0,
        token_importance=True
    ):
        # Parameter validation
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")

        defaults = dict(
            lr=lr,
            betas=betas,
            history_size=history_size,
            eps=eps,
            weight_decay=weight_decay
        )
        super(SimpsonOptimizer, self).__init__(params, defaults)

        self.token_importance = token_importance
        self.importance_scores = None

    def set_importance_scores(self, scores):
        """Set token importance scores from ABLT tokenizer"""
        self.importance_scores = torch.tensor(scores, dtype=torch.float32)

    def apply_simpson_rule(self, gradients):
        """Advanced Simpson's Rule Integration"""
        n = len(gradients)
        if n < 3:
            return gradients[-1]

        # Composite Simpson's rule with adaptive weighting
        integrated_grad = gradients[0] + gradients[-1]

        # Weighted integration
        for i in range(1, n, 2):
            integrated_grad += 4 * gradients[i]

        for i in range(2, n-1, 2):
            integrated_grad += 2 * gradients[i]

        return integrated_grad / (3 * ((n-1) // 2))

    def step(self, closure=None):
        """Adaptive Gradient Update with Simpson Integration"""
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data

                # Weight decay
                if group['weight_decay'] != 0:
                    grad = grad.add(p.data, alpha=group['weight_decay'])

                # Parameter state management
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['grad_history'] = []
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_avg_sq'] = torch.zeros_like(p.data)

                # Token importance weighting
                if self.token_importance and self.importance_scores is not None:
                    if grad.dim() >= 2 and grad.size(0) == self.importance_scores.size(0):
                        importance_weights = self.importance_scores.view(-1, 1)
                        while importance_weights.dim() < grad.dim():
                            importance_weights = importance_weights.unsqueeze(-1)
                        importance_weights = importance_weights.expand_as(grad)
                        grad = grad * importance_weights

                # Gradient history management
                state['grad_history'].append(grad.clone())
                if len(state['grad_history']) > group['history_size']:
                    state['grad_history'].pop(0)

                state['step'] += 1
                beta1, beta2 = group['betas']

                # Momentum and adaptive learning rate
                state['exp_avg'].mul_(beta1).add_(grad, alpha=1 - beta1)
                state['exp_avg_sq'].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Bias correction
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']

                # Simpson integration for gradient updates
                if len(state['grad_history']) >= 3:
                    integrated_grad = self.apply_simpson_rule(state['grad_history'])

                    denom = state['exp_avg_sq'].sqrt().add_(group['eps'])
                    step_size = group['lr'] * math.sqrt(bias_correction2) / bias_correction1

                    p.data.addcdiv_(integrated_grad, denom, value=-step_size)

        return loss

In [ ]:
# Load ABLT tokenizer
ablt = ABLT.load("ablt_tokenizer.pkl")

# Example text
text = "Abantu balina okuyiga oluganda."
token_ids = ablt.encode(text)
_, importance_scores = ablt.get_tokens_with_importance(text)

# Convert to tensors
input_tensor = torch.tensor([token_ids])
importance_tensor = torch.tensor(importance_scores)

# Define a sample model
class AlkebulanModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, embedding_dim, batch_first=True)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        return self.output_layer(lstm_out)

# Initialize model and optimizer
model = AlkebulanModel(ablt.sp_tokenizer.get_piece_size())
optimizer = SimpsonOptimizer(model.parameters(), lr=0.001)

# Set importance scores
optimizer.set_importance_scores(importance_scores)

# Training loop (simplified)
criterion = nn.CrossEntropyLoss()
target = torch.randint(0, ablt.sp_tokenizer.get_piece_size(), input_tensor.shape)

# Training step
optimizer.zero_grad()
outputs = model(input_tensor)
loss = criterion(outputs.view(-1, ablt.sp_tokenizer.get_piece_size()), target.view(-1))
loss.backward()
optimizer.step()

print(f"Loss: {loss.item()}")

NameError: name 'torch' is not defined

In [ ]:
import pandas as pd
import sentencepiece as spm
import pickle
import torch
import torch.nn as nn
import math
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Existing ABLT and SimpsonOptimizer classes remain the same

class LugandaParallelDataset(Dataset):
    def __init__(self, csv_path, ablt_tokenizer):
        """
        Initialize Luganda Parallel Dataset

        Args:
        - csv_path: Path to parallel corpus CSV
        - ablt_tokenizer: ABLT tokenizer for processing
        """
        self.data = pd.read_csv(csv_path)
        self.tokenizer = ablt_tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        luganda_text = self.data.iloc[idx]['luganda']
        english_text = self.data.iloc[idx]['english']

        # Tokenize and encode
        luganda_tokens = self.tokenizer.encode(luganda_text)
        english_tokens = self.tokenizer.encode(english_text)

        return {
            'luganda_input': torch.tensor(luganda_tokens),
            'english_target': torch.tensor(english_tokens)
        }

def create_dataloader(csv_path, ablt_tokenizer, batch_size=32):
    dataset = LugandaParallelDataset(csv_path, ablt_tokenizer)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

def train_epoch(model, dataloader, optimizer, criterion, device, ablt_tokenizer):
    model.train()
    total_loss = 0

    for batch in dataloader:
        luganda_input = batch['luganda_input'].to(device)
        english_target = batch['english_target'].to(device)

        # Get importance scores for the first sequence in the batch
        _, importance_scores = ablt_tokenizer.get_tokens_with_importance(
            ablt_tokenizer.decode(luganda_input[0].cpu().numpy().tolist())
        )

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(luganda_input)

        # Compute loss
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            english_target.view(-1)
        )

        # Backward pass
        loss.backward()

        # Update with Simpson Optimizer
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def main():
    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load tokenizer
    ablt = ABLT.load("ablt_tokenizer.pkl")

    # Create dataset and dataloader
    dataloader = create_dataloader('luganda-approx50k.csv', ablt)

    # Create model
    vocab_size = ablt.sp_tokenizer.get_piece_size()
    model = SimpleModel(vocab_size).to(device)

    # Optimizer and Loss
    optimizer = SimpsonOptimizer(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # Training
    num_epochs = 10
    for epoch in range(num_epochs):
        loss = train_epoch(model, dataloader, optimizer, criterion, device, ablt)
        print(f"Epoch {epoch+1}, Loss: {loss}")

if __name__ == "__main__":
    main()

ParserError: Error tokenizing data. C error: Expected 2 fields in line 144, saw 3


In [ ]:
import pandas as pd
import csv

# Read the CSV file
def read_clean_csv(filename):
    # Try reading with quote handling
    try:
        df = pd.read_csv(filename, quotechar='"', escapechar='\\')
        print("DataFrame shape:", df.shape)
        print("\nColumns:", list(df.columns))
        print("\nFirst few rows:")
        print(df.head())
        return df
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# Manually clean the CSV if needed
def clean_csv(input_filename, output_filename):
    with open(input_filename, 'r', encoding='utf-8') as infile, \
         open(output_filename, 'w', encoding='utf-8', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)

        # Track and skip problematic lines
        for i, row in enumerate(reader, 1):
            try:
                # Ensure exactly 2 columns
                if len(row) == 2:
                    writer.writerow(row)
                elif len(row) > 2:
                    print(f"Skipping line {i} with {len(row)} columns")
                    # Optionally, merge extra columns if needed
                    # writer.writerow([row[0], ' '.join(row[1:])])
            except Exception as e:
                print(f"Error processing line {i}: {e}")

# Clean the CSV first
clean_csv('luganda-approx50k.csv', 'cleaned_luganda_corpus.csv')

# Then read the cleaned file
df = read_clean_csv('cleaned_luganda_corpus.csv')

Skipping line 144 with 3 columns
Skipping line 331 with 3 columns
Skipping line 332 with 3 columns
Skipping line 420 with 3 columns
Skipping line 607 with 3 columns
Skipping line 659 with 3 columns
Skipping line 904 with 3 columns
Skipping line 1013 with 3 columns
Skipping line 1078 with 3 columns
Skipping line 1323 with 3 columns
Skipping line 1432 with 3 columns
Skipping line 1467 with 3 columns
Skipping line 1955 with 3 columns
Skipping line 1959 with 3 columns
Skipping line 2041 with 3 columns
Skipping line 2634 with 3 columns
Skipping line 2999 with 3 columns
Skipping line 3015 with 3 columns
Skipping line 3032 with 3 columns
Skipping line 3129 with 3 columns
Skipping line 3364 with 3 columns
Skipping line 3367 with 3 columns
Skipping line 3662 with 3 columns
Skipping line 3785 with 3 columns
Skipping line 3936 with 3 columns
Skipping line 4041 with 3 columns
Skipping line 4471 with 3 columns
Skipping line 4485 with 3 columns
Skipping line 4626 with 3 columns
Skipping line 4759 wi

In [ ]:
import pandas as pd
import csv

def read_clean_csv(filename):
    try:
        # Use more robust CSV reading parameters
        df = pd.read_csv(filename,
                         quotechar='"',
                         escapechar='\\',
                         encoding='utf-8',
                         dtype=str)  # Ensure all columns are read as strings

        # Basic dataset information
        print("Dataset Overview:")
        print(f"Total Rows: {len(df)}")
        print(f"Columns: {list(df.columns)}")

        # Check for any missing values
        print("\nMissing Values:")
        print(df.isnull().sum())

        # Sample data
        print("\nFirst 5 Rows:")
        print(df.head())

        # Basic text length statistics
        print("\nText Length Statistics:")
        df['english_length'] = df['English'].str.len()
        df['luganda_length'] = df['Luganda'].str.len()
        print("English Text Length:")
        print(df['english_length'].describe())
        print("\nLuganda Text Length:")
        print(df['luganda_length'].describe())

        return df

    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# Clean the CSV first
def clean_csv(input_filename, output_filename):
    with open(input_filename, 'r', encoding='utf-8') as infile, \
         open(output_filename, 'w', encoding='utf-8', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)

        # Track and skip problematic lines
        for i, row in enumerate(reader, 1):
            try:
                # Ensure exactly 2 columns
                if len(row) == 2:
                    writer.writerow(row)
                elif len(row) > 2:
                    print(f"Skipping line {i} with {len(row)} columns")
            except Exception as e:
                print(f"Error processing line {i}: {e}")

# Clean the CSV first
clean_csv('luganda-approx50k.csv', 'cleaned_luganda_corpus.csv')

# Then read the cleaned file
df = read_clean_csv('cleaned_luganda_corpus.csv')

Skipping line 144 with 3 columns
Skipping line 331 with 3 columns
Skipping line 332 with 3 columns
Skipping line 420 with 3 columns
Skipping line 607 with 3 columns
Skipping line 659 with 3 columns
Skipping line 904 with 3 columns
Skipping line 1013 with 3 columns
Skipping line 1078 with 3 columns
Skipping line 1323 with 3 columns
Skipping line 1432 with 3 columns
Skipping line 1467 with 3 columns
Skipping line 1955 with 3 columns
Skipping line 1959 with 3 columns
Skipping line 2041 with 3 columns
Skipping line 2634 with 3 columns
Skipping line 2999 with 3 columns
Skipping line 3015 with 3 columns
Skipping line 3032 with 3 columns
Skipping line 3129 with 3 columns
Skipping line 3364 with 3 columns
Skipping line 3367 with 3 columns
Skipping line 3662 with 3 columns
Skipping line 3785 with 3 columns
Skipping line 3936 with 3 columns
Skipping line 4041 with 3 columns
Skipping line 4471 with 3 columns
Skipping line 4485 with 3 columns
Skipping line 4626 with 3 columns
Skipping line 4759 wi

In [ ]:
import sentencepiece as spm
import torch
from transformers import AutoTokenizer

class MultilingualTokenizer:
    def __init__(self, luganda_tokenizer, english_tokenizer):
        """
        Initialize with separate tokenizers for Luganda and English

        Args:
        - luganda_tokenizer: ABLT Tokenizer for Luganda
        - english_tokenizer: Hugging Face or other English tokenizer
        """
        self.luganda_tokenizer = luganda_tokenizer
        self.english_tokenizer = english_tokenizer

    def tokenize_luganda(self, text):
        """Tokenize Luganda text using ABLT tokenizer"""
        return self.luganda_tokenizer.tokenize_text(text)

    def tokenize_english(self, text):
        """Tokenize English text using pre-trained English tokenizer"""
        return self.english_tokenizer.tokenize(text)

    def encode_luganda(self, text):
        """Encode Luganda text"""
        return self.luganda_tokenizer.encode(text)

    def encode_english(self, text):
        """Encode English text"""
        return self.english_tokenizer.encode(text)

# Example setup
def create_multilingual_tokenizer(ablt_tokenizer):
    # Choose an appropriate English tokenizer
    english_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    return MultilingualTokenizer(
        luganda_tokenizer=ablt_tokenizer,
        english_tokenizer=english_tokenizer
    )

# Create the multilingual tokenizer
multilingual_tokenizer = create_multilingual_tokenizer(ablt)

# Demonstration
def demonstrate_tokenization(text_pairs):
    print("🌍 Multilingual Tokenization Comparison:")
    for english, luganda in text_pairs:
        print("\n--- Tokenization ---")
        print("English Original:", english)
        print("English Tokens:", multilingual_tokenizer.tokenize_english(english))
        print("English Token IDs:", multilingual_tokenizer.encode_english(english))

        print("\nLuganda Original:", luganda)
        print("Luganda Tokens:", multilingual_tokenizer.tokenize_luganda(luganda))
        print("Luganda Token IDs:", multilingual_tokenizer.encode_luganda(luganda))

# Sample text pairs for demonstration
sample_texts = [
    (df['English'].iloc[0], df['Luganda'].iloc[0]),
    (df['English'].iloc[1], df['Luganda'].iloc[1]),
    (df['English'].iloc[2], df['Luganda'].iloc[2])
]

demonstrate_tokenization(sample_texts)
validate_tokenization(ablt)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

🌍 Multilingual Tokenization Comparison:

--- Tokenization ---
English Original: It's the government's responsibility to teach its people about various diseases
English Tokens: ['it', "'", 's', 'the', 'government', "'", 's', 'responsibility', 'to', 'teach', 'its', 'people', 'about', 'various', 'diseases']
English Token IDs: [101, 2009, 1005, 1055, 1996, 2231, 1005, 1055, 5368, 2000, 6570, 2049, 2111, 2055, 2536, 7870, 102]

Luganda Original: Buvunaanyizibwa bwa gavumenti okusomesa abantu baayo ku bulwadde obw'enjawulo
Luganda Tokens: ['▁B', '▁uvu', 'naa', 'nyi', 'zi', '▁b', 'wa', '▁b', 'wa', '▁', 'ga', 'vum', 'en', 'ti', '▁', 'oku', '▁somes', '▁a', '▁', 'aba', '▁n', 'tu', '▁', 'ba', '▁ayo', '▁', 'ku', '▁b', '▁u', 'lw', '▁', 'add', '▁e', '▁', 'obw', "▁'", 'en', 'jawu', '▁lo']
Luganda Token IDs: [450, 889, 21, 467, 17, 376, 20, 376, 20, 367, 105, 442, 85, 400, 367, 64, 545, 368, 367, 51, 371, 65, 367, 70, 518, 367, 76, 376, 386, 45, 367, 453, 370, 367, 40, 379, 85, 544, 468]

--- Tokeniza

NameError: name 'validate_tokenization' is not defined

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

def analyze_tokenizers(ablt_tokenizer, english_tokenizer, csv_path, sample_size=100):
    # Load the dataset
    df = pd.read_csv(csv_path)

    # Random sampling
    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42)

    def tokenize_and_analyze(text_series, tokenizer, language):
        all_tokens = []
        all_token_lengths = []

        # For ABLT tokenizer, we'll also track importance
        if hasattr(tokenizer, 'get_tokens_with_importance'):
            all_token_importances = []

        for text in text_series:
            # Tokenize
            if hasattr(tokenizer, 'tokenize_text'):
                tokens = tokenizer.tokenize_text(text)
            else:
                tokens = tokenizer.tokenize(text)

            all_tokens.extend(tokens)

            # Token lengths
            all_token_lengths.extend([len(token) for token in tokens])

            # Token importance (for ABLT)
            if hasattr(tokenizer, 'get_tokens_with_importance'):
                try:
                    _, importances = tokenizer.get_tokens_with_importance(text)
                    all_token_importances.extend(importances)
                except Exception as e:
                    print(f"Error processing {language} text: {e}")

        # Prepare analysis results
        analysis = {
            'total_tokens': len(all_tokens),
            'unique_tokens': len(set(all_tokens)),
            'token_lengths': {
                'min': min(all_token_lengths) if all_token_lengths else 0,
                'max': max(all_token_lengths) if all_token_lengths else 0,
                'mean': np.mean(all_token_lengths) if all_token_lengths else 0,
                'median': np.median(all_token_lengths) if all_token_lengths else 0
            },
            'most_common_tokens': Counter(all_tokens).most_common(10)
        }

        # Add importance stats if available
        if hasattr(tokenizer, 'get_tokens_with_importance'):
            analysis['token_importances'] = {
                'mean': np.mean(all_token_importances) if all_token_importances else 0,
                'median': np.median(all_token_importances) if all_token_importances else 0,
                'max': np.max(all_token_importances) if all_token_importances else 0,
                'min': np.min(all_token_importances) if all_token_importances else 0
            }

        return analysis

    # Analyze both languages
    luganda_analysis = analyze_tokenizers(ablt_tokenizer, ablt_tokenizer, csv_path, sample_size)
    english_analysis = analyze_tokenizers(ablt_tokenizer, english_tokenizer, csv_path, sample_size)

    # Visualization
    plt.figure(figsize=(20, 15))

    # Token Length Comparison
    plt.subplot(2, 2, 1)
    plt.title('Token Length Distribution')
    plt.hist([len(token) for token in luganda_analysis['tokens']], alpha=0.5, label='Luganda')
    plt.hist([len(token) for token in english_analysis['tokens']], alpha=0.5, label='English')
    plt.xlabel('Token Length')
    plt.ylabel('Frequency')
    plt.legend()

    # Most Common Tokens
    plt.subplot(2, 2, 2)
    plt.title('Most Common Tokens')
    lug_tokens, lug_counts = zip(*luganda_analysis['most_common_tokens'])
    plt.bar(lug_tokens, lug_counts, label='Luganda', alpha=0.5)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Tokens')
    plt.ylabel('Frequency')

    # Detailed Linguistic Analysis
    print("🔬 Comprehensive Tokenization Analysis:\n")

    print("Luganda Tokenization:")
    print(f"Total Tokens: {luganda_analysis['total_tokens']}")
    print(f"Unique Tokens: {luganda_analysis['unique_tokens']}")
    print("\nToken Length Statistics:")
    for k, v in luganda_analysis['token_lengths'].items():
        print(f"  {k.capitalize()}: {v:.2f}")

    print("\nMost Common Luganda Tokens:")
    for token, count in luganda_analysis['most_common_tokens']:
        print(f"  {token}: {count}")

    print("\n" + "="*50 + "\n")

    print("English Tokenization:")
    print(f"Total Tokens: {english_analysis['total_tokens']}")
    print(f"Unique Tokens: {english_analysis['unique_tokens']}")
    print("\nToken Length Statistics:")
    for k, v in english_analysis['token_lengths'].items():
        print(f"  {k.capitalize()}: {v:.2f}")

    print("\nMost Common English Tokens:")
    for token, count in english_analysis['most_common_tokens']:
        print(f"  {token}: {count}")

    # Linguistic Complexity Indicators
    print("\n🧠 Linguistic Complexity Indicators:")
    print("Luganda:")
    print(f"  Average Token Length: {np.mean([len(token) for token in luganda_analysis['tokens']]):.2f}")
    print(f"  Morphological Diversity: {len(luganda_analysis['unique_tokens']) / len(luganda_analysis['tokens']):.4f}")

    print("\nEnglish:")
    print(f"  Average Token Length: {np.mean([len(token) for token in english_analysis['tokens']]):.2f}")
    print(f"  Morphological Diversity: {len(english_analysis['unique_tokens']) / len(english_analysis['tokens']):.4f}")

    # Save visualization
    plt.tight_layout()
    plt.savefig('tokenizer_analysis.png')
    plt.close()

    return luganda_analysis, english_analysis

# Prepare tokenizers
english_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Run the analysis
luganda_results, english_results = analyze_tokenizers(
    ablt,
    english_tokenizer,
    'cleaned_luganda_corpus.csv',
    sample_size=100
)

In [ ]:
# Create a dictionary with all the necessary components
tokenizer_package = {
    'model_path': ablt.model_path,
    'morphological_tokenizer_data': {
        'linguistic_features': ablt.morphological_tokenizer.linguistic_features,
        'phonological_rules': ablt.morphological_tokenizer.phonological_rules
    }
}

# Save the package
with open('ablt_v1_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer_package, f)

print("Saved ABLT tokenizer components to 'ablt_v1_tokenizer.pkl'")

Saved ABLT tokenizer components to 'ablt_v1_tokenizer.pkl'
